# Uji Validitas dan Reliabilitas Instrumen
Statistika Probabilitas, Kelas 3 TI C

Kode mengikuti Langkah 1 sampai 8 pada modul. Hanya ada dua penyesuaian karena bentuk data:
1. Kolom *Durasi penggunaan gadget* (bukan skala Likert) dikeluarkan dari butir (Langkah 3).
2. Label `Sangat Tidak Setuju (STJ)` ditambahkan ke kamus skala karena di berkas Excel tertulis STJ, bukan STS (Langkah 4).

## Langkah 1: Persiapan Lingkungan dan Impor Pustaka

In [ ]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

## Langkah 2: Unggah Berkas Data

In [ ]:
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

## Langkah 3: Membaca Data dan Memisahkan Butir

In [ ]:
JUMLAH_KOLOM_DEMOGRAFI = 5  # Timestamp, Kelas, Jenis Kelamin, 2 kolom jumlah platform AI
df = pd.read_excel(nama_file)
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]

# Penyesuaian: kolom Durasi gadget ada di tengah dan bukan butir Likert, jadi dikeluarkan
butir = butir.drop(columns=[c for c in butir.columns if "[" not in c])

print("Jumlah responden :", len(df))
print("Jumlah butir :", butir.shape[1])
print("Data kosong :", int(butir.isna().sum().sum()))

## Langkah 4: Mengubah Jawaban Likert Menjadi Angka

In [ ]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Sangat Tidak Setuju (STJ)": 1,  # penyesuaian: di berkas Excel tertulis STJ
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}
num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
num.head()

## Langkah 5: Mengelompokkan Butir per Dimensi

In [ ]:
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?) \[", kolom).group(1).strip()
    dimensi.setdefault(nama, []).append(kolom)
for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

## Langkah 6: Uji Validitas (Korelasi Pearson Butir-Total)

In [ ]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

## Langkah 7: Uji Reliabilitas (Cronbach's Alpha)

In [ ]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori(a),
    })
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3),
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

## Langkah 8: Menggabungkan dan Mengunduh Hasil

In [ ]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")
with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)
files.download("hasil_uji_validitas_reliabilitas.xlsx")

## Uji ulang setelah butir tidak valid dibuang (Latihan 3)

In [ ]:
buang = gabung[gabung["Keterangan"] == "Tidak valid"]
for nama, daftar in dimensi.items():
    kolom_buang = [daftar[int(b[1:]) - 1] for b in buang.loc[buang["Dimensi"] == nama, "Butir"]]
    if kolom_buang:
        sisa = [k for k in daftar if k not in kolom_buang]
        print(f"{nama}: alpha {cronbach_alpha(num[daftar]):.3f} -> {cronbach_alpha(num[sisa]):.3f}")